# California / Hawaii NFDRS Normals 2016–2025 — by RAWS, PSA and FDRA

**No login, no API key.** FEMS climatology downloads are public. This notebook pulls them in
**batches of 20 stations per request, one year at a time**, and caches every batch in `fems_raw/`.
If anything stops, run the cell again and it picks up where it left off.

Everything is in this one file: cells 3 and 4 write the helper code (`ca_normals.py`) and the FDRA
station list (`fdra_station_membership_2026.csv`) next to the notebook. Put the notebook in its own
folder and choose **Run All**.

- **Stations:** the RAWS in the live *CA NFDRS 3.0 (FEMS)* layer (California + Hawaii), fuel model Y.
- **FDRAs (national forests):** each FDRA's FDOP stations and fuel model from the 2026 Forest Thresholds sheet.
  ENF 335 has peak and non-peak station sets, so it gets two units.
- **Data:** FEMS daily NFDR summaries, **observed** values only; plus daily max temperature and min RH.
- **Indices:** ERC, IC, BI, SC, 1/10/100/1000-hr FM, KBDI, GSI, woody and herb FM.

Outputs land in `output/` (CSVs) and `output/site_data/` (files for the FDRA website).

In [ ]:
# 1 — Install packages (once)
import subprocess, sys
for pkg in ['pandas', 'numpy', 'requests', 'matplotlib']:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'], capture_output=True, text=True)
    print(('OK  ' if r.returncode == 0 else 'FAIL') + f' {pkg}')

In [ ]:
# 2 — Settings  (edit these if you like)
from pathlib import Path
START_YEAR = 2016
END_YEAR   = 2025
BATCH      = 20          # stations per FEMS request
WORKERS    = 4           # requests at once (keep it small; FEMS is a shared service)

HERE    = Path.cwd()
RAW_DIR = HERE / 'fems_raw'      # cached downloads
OUT_DIR = HERE / 'output'
FDRA_MEMBERSHIP = HERE / 'fdra_station_membership_2026.csv'
INCLUDE_STATION_DOY = False      # day-of-year normals per station (large file)
print('Working in', HERE)

In [ ]:
%%writefile ca_normals.py
"""
ca_normals.py
─────────────
California NFDRS normals (default 2016–2025) by RAWS station, PSA and FDRA.

Station list comes from the live "CA NFDRS 3.0 (FEMS)" layer (California + Hawaii) that the
Experience Builder app displays (oncc_oscc_scripting, NIFC org), so the
normals line up 1:1 with the live values: same stations, same PSA
assignments, same fuel model (Y).

Pipeline
  1. stations  ← CA NFDRS 3.0 RAWS layer (public REST; CA + HI)
  2. FDRAs     ← FDOP stations + fuel model per FDRA (2026 Forest Thresholds sheet)
  3. download  ← FEMS download-nfdr-daily-summary (public, no key), batches of 20
                 stations per request per year, cached so re-runs only fetch what's missing
  4. aggregate ← observed (NFDRType=O) daily values → PSA / FDRA daily means
                 (same rule as the dashboard: average reporting stations)
  5. normals   ← annual percentiles + day-of-year normals per unit & index
  6. outputs   ← CSVs ready to add to ArcGIS Online as hosted tables

Used by CA_Normals_2016_2025.ipynb — run the notebook, not this file.
"""

from __future__ import annotations

import re
import json
import logging
import math
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import requests

log = logging.getLogger('ca_normals')

CA_NFDRS_URL = ('https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/'
                'CA_NFDRS/FeatureServer')
FEMS_URL = 'https://fems.fs2c.usda.gov/api/ext-climatology/download-nfdr-daily-summary/'

# index key → FEMS CSV column
FIELD_COLS = {
    'erc': 'ERC', 'ic': 'IC', 'bi': 'BI', 'sc': 'SC',
    'fm1': '1HrFM', 'fm10': '10HrFM', 'fm100': '100HrFM', 'fm1000': '1000HrFM',
    'kbdi': 'KBDI', 'gsi': 'GSI', 'woody': 'WoodyFM', 'herb': 'HerbFM',
}
INDICES = list(FIELD_COLS)
MOISTURE = {'fm1', 'fm10', 'fm100', 'fm1000', 'woody', 'herb'}
# Annual percentiles: dry-side lows matter for fuel moisture, highs for indices.
ANNUAL_PCTS = [3, 5, 10, 20, 50, 80, 90, 95, 97]
DOY_STATS = ['mean', 'p10', 'p25', 'p50', 'p75', 'p90']
DOY_WINDOW = 7          # ± days pooled around each day-of-year (15-day window)


# ── 1. Stations ───────────────────────────────────────────────────────────────
def load_ca_stations(exclude_gaccs=()) -> pd.DataFrame:
    url = f'{CA_NFDRS_URL}/0/query'
    params = dict(where='1=1', outFields='stationId,StationName,GACC,PSA,FuelModel,Latitude,Longitude,Unit,Agency',
                  returnGeometry='false', f='json', resultRecordCount=2000)
    feats = requests.get(url, params=params, timeout=60).json()['features']
    df = pd.DataFrame([f['attributes'] for f in feats])
    df = df[~df['GACC'].isin(exclude_gaccs)].copy()
    df['stationId'] = df['stationId'].astype(int)
    df['FuelModel'] = df['FuelModel'].fillna('Y')
    log.info('CA stations: %d (%s)', len(df), df['GACC'].value_counts().to_dict())
    return df.reset_index(drop=True)


def load_ca_psas() -> pd.DataFrame:
    url = f'{CA_NFDRS_URL}/1/query'
    feats = requests.get(url, params=dict(where='1=1', outFields='PSANationalCode,PSAName,GACCUnitID',
                                          returnGeometry='false', f='json'), timeout=60).json()['features']
    return pd.DataFrame([f['attributes'] for f in feats])


# ── 2. FDRA station lists (2026 Forest Thresholds sheet) ──────────────────────
def load_fdra_membership(csv_path) -> tuple[dict, pd.DataFrame]:
    """FDOP station lists from the 2026 Forest Thresholds sheet
    (fdra_station_membership_2026.csv from align_fdras_to_sheet.py).

    Returns ({fdra_id: [(stationId, fuel_model), ...]}, pairs DataFrame with
    stationId / FuelModel for download). Seasonal FDRAs (e.g. ENF 335 peak /
    non-peak) become separate units: 'ENF-335 (peak)'.
    """
    m = pd.read_csv(csv_path).fillna('')
    m['unit'] = np.where(m['season'] != '', m['fdra_id'] + ' (' + m['season'] + ')', m['fdra_id'])
    mem = {u: list(zip(g['stationId'].astype(int), g['fuel_model'])) for u, g in m.groupby('unit')}
    pairs = m[['stationId', 'fuel_model']].rename(columns={'fuel_model': 'FuelModel'}).drop_duplicates()
    log.info('FDRA membership: %d FDRAs, %d station × fuel-model pairs', len(mem), len(pairs))
    return mem, pairs


# ── 3. FEMS download (public, batched, cached) ────────────────────────────────
# FEMS ext-climatology downloads are public (no API key). One request takes many
# stations and several fuel models: e.g. 20 stations × 3 fuel models × 1 year ≈ 3 s.
FEMS_WX_URL = 'https://fems.fs2c.usda.gov/api/ext-climatology/download-wx-daily-summary/'
BATCH = 20


def _fems_get(url, ids, start, end, fuel_models=None, dataset='observation', retries=4):
    params = dict(dataset=dataset, startDate=start, endDate=end, dataFormat='csv',
                  stationIds=','.join(map(str, ids)))
    if fuel_models:
        params['fuelModels'] = ','.join(fuel_models)
    for a in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, headers={'Accept': 'text/csv'}, timeout=300)
            if r.status_code == 404:
                return ''
            r.raise_for_status()
            return r.text
        except Exception:
            if a == retries:
                raise
            time.sleep(5 * a)


def _batches(ids, n=None):
    n = n or BATCH
    ids = sorted({int(i) for i in ids})
    return [ids[i:i + n] for i in range(0, len(ids), n)]


def _tag(ids):
    import hashlib
    return hashlib.md5(','.join(map(str, ids)).encode()).hexdigest()[:8]


def _download(url, prefix, ids, fuel_models, start_year, end_year, raw_dir, workers):
    raw_dir.mkdir(parents=True, exist_ok=True)
    jobs = []
    for chunk in _batches(ids):
        for y in range(start_year, end_year + 1):
            f = raw_dir / f'{prefix}_{y}_{_tag(chunk)}.csv'
            if not f.exists():
                jobs.append((chunk, y, f))
    log.info('FEMS %s: %d batch-years to download', prefix, len(jobs))
    failed = []

    def one(job):
        chunk, y, f = job
        txt = _fems_get(url, chunk, f'{y}-01-01', f'{y}-12-31', fuel_models)
        f.write_text(txt or '', encoding='utf-8')

    with ThreadPoolExecutor(max_workers=workers) as pool:
        futs = {pool.submit(one, j): j for j in jobs}
        for i, fut in enumerate(as_completed(futs), 1):
            try:
                fut.result()
            except Exception as e:
                failed.append((futs[fut][1], str(e)[:120]))
            if i % 25 == 0:
                log.info('  %d / %d', i, len(jobs))
    if failed:
        log.warning('%d batch-years failed (re-run to retry): %s', len(failed), failed[:5])
    return failed


def download_history(stations: pd.DataFrame, start_year: int, end_year: int, raw_dir: Path,
                     workers: int = 4):
    """Batched, cached NFDR daily summaries (observed). `stations` needs stationId and
    FuelModel; all fuel models in use are requested for every station (one call per
    batch of 20 stations per year). Public FEMS endpoint — no login."""
    fms = sorted(stations['FuelModel'].dropna().astype(str).str.upper().unique())
    return _download(FEMS_URL, 'nfdr_' + ''.join(fms), stations['stationId'], fms,
                     start_year, end_year, raw_dir, workers)


def load_history(stations: pd.DataFrame, start_year: int, end_year: int, raw_dir: Path) -> pd.DataFrame:
    """Observed daily values: columns stationId, fm, date, <indices>."""
    want = set(zip(stations['stationId'].astype(int), stations['FuelModel'].astype(str).str.upper()))
    frames = []
    for f in sorted(raw_dir.glob('nfdr_*.csv')):
        if f.stat().st_size < 50:
            continue
        df = pd.read_csv(f)
        df.columns = df.columns.str.strip()
        if 'ObservationTime' not in df.columns:
            continue
        df = df[df['NFDRType'].astype(str).str.upper() == 'O']
        out = pd.DataFrame({'stationId': pd.to_numeric(df['StationId'], errors='coerce'),
                            'fm': df['FuelModel'].astype(str).str.upper(),
                            'date': pd.to_datetime(df['ObservationTime'], errors='coerce').dt.normalize()})
        for k, col in FIELD_COLS.items():
            out[k] = pd.to_numeric(df[col], errors='coerce') if col in df.columns else np.nan
        frames.append(out.dropna(subset=['date', 'stationId']))
    hist = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=['stationId', 'fm', 'date', *INDICES])
    hist['stationId'] = hist['stationId'].astype(int)
    hist = hist[[p in want for p in zip(hist['stationId'], hist['fm'])]]
    hist = hist[hist['date'].dt.year.between(start_year, end_year)]
    hist = hist.groupby(['stationId', 'fm', 'date'], as_index=False)[INDICES].mean()
    for k in INDICES:                       # FEMS uses negatives for missing
        hist.loc[hist[k] < 0, k] = np.nan
    log.info('History: %d station-days, %d stations, %s → %s', len(hist), hist['stationId'].nunique(),
             hist['date'].min(), hist['date'].max())
    return hist


# ── 4. Aggregate to PSA / FDRA ────────────────────────────────────────────────
def unit_daily(hist: pd.DataFrame, membership: dict, unit_type: str, fuel_model='Y') -> pd.DataFrame:
    """Daily mean of member stations for each unit (non-reporting stations ignored).

    membership: {unit_id: [stationId, ...]} using `fuel_model` for every unit, or
                {unit_id: [(stationId, fuel_model), ...]} for per-unit fuel models (FDRAs).
    """
    parts = []
    for uid, members in membership.items():
        pairs = [m if isinstance(m, tuple) else (m, fuel_model) for m in members]
        key = pd.MultiIndex.from_tuples(pairs, names=['stationId', 'fm'])
        sub = hist[pd.MultiIndex.from_frame(hist[['stationId', 'fm']]).isin(key)]
        if sub.empty:
            continue
        g = sub.groupby('date')[INDICES].mean()
        g['n_reporting'] = sub.groupby('date')['stationId'].nunique()
        g = g.reset_index()
        g.insert(0, 'unit_id', uid)
        g.insert(0, 'unit_type', unit_type)
        parts.append(g)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


# ── 5. Normals ────────────────────────────────────────────────────────────────
def annual_normals(daily: pd.DataFrame) -> pd.DataFrame:
    """One row per unit: n_days + mean and percentiles of every index (full year)."""
    rows = []
    for (ut, uid), g in daily.groupby(['unit_type', 'unit_id']):
        r = {'unit_type': ut, 'unit_id': uid, 'n_days': len(g),
             'first_date': g['date'].min().date().isoformat(), 'last_date': g['date'].max().date().isoformat()}
        for k in INDICES:
            v = g[k].dropna().to_numpy()
            r[f'{k}_n'] = len(v)
            r[f'{k}_mean'] = round(float(v.mean()), 2) if len(v) else None
            for p in ANNUAL_PCTS:
                r[f'{k}_p{p}'] = round(float(np.percentile(v, p)), 2) if len(v) else None
        rows.append(r)
    return pd.DataFrame(rows)


def doy_normals(daily: pd.DataFrame, plot_year: int) -> pd.DataFrame:
    """Day-of-year normals (± DOY_WINDOW days, all years pooled).

    One row per unit × day (1–365; Feb 29 folded into Feb 28). `plot_date`
    puts every row on the same calendar year so Experience Builder line charts
    can overlay normals on the current season.
    """
    import warnings
    d = daily.copy()
    doy = d['date'].dt.dayofyear
    leap = d['date'].dt.is_leap_year
    d['doy'] = np.where(leap & (doy > 59), doy - 1, doy)      # 1..365
    base = pd.Timestamp(f'{plot_year}-01-01')
    days = np.arange(1, 366)
    shifts = range(-DOY_WINDOW, DOY_WINDOW + 1)
    frames = []
    for (ut, uid), g in d.groupby(['unit_type', 'unit_id']):
        out = pd.DataFrame({'unit_type': ut, 'unit_id': uid, 'doy': days,
                            'plot_date': base + pd.to_timedelta(days - 1, unit='D') + pd.Timedelta(hours=12),
                            'month_day': (base + pd.to_timedelta(days - 1, unit='D')).strftime('%m-%d')})
        rank = g.groupby('doy').cumcount().to_numpy()          # nth value for that doy
        width = int(rank.max()) + 1 if len(rank) else 1
        for k in INDICES:
            M = np.full((365, width), np.nan)
            M[g['doy'].to_numpy() - 1, rank] = g[k].to_numpy()
            W = np.concatenate([np.roll(M, -o, axis=0) for o in shifts], axis=1)   # (365, 15*width)
            with warnings.catch_warnings():
                warnings.simplefilter('ignore', RuntimeWarning)  # all-NaN windows → NaN
                out[f'{k}_mean'] = np.round(np.nanmean(W, axis=1), 2)
                q = np.nanpercentile(W, [10, 25, 50, 75, 90], axis=1)
            for i, name in enumerate(['p10', 'p25', 'p50', 'p75', 'p90']):
                out[f'{k}_{name}'] = np.round(q[i], 2)
        frames.append(out)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


# ── 6. Write outputs ──────────────────────────────────────────────────────────
def write_outputs(out_dir: Path, start_year: int, end_year: int, stations, fdra_tbl,
                  annual: pd.DataFrame, doy: pd.DataFrame):
    out_dir.mkdir(parents=True, exist_ok=True)
    tag = f'{start_year}_{end_year}'
    files = {}
    meta = {'period': f'{start_year}-{end_year}', 'generated': date.today().isoformat(),
            'source': 'FEMS download-nfdr-daily-summary, observed (NFDRType=O) daily values',
            'stations_from': 'CA NFDRS 3.0 (FEMS) RAWS layer (California + Hawaii)',
            'fuel_model': sorted(stations['FuelModel'].unique().tolist()),
            'aggregation': 'unit daily value = mean of reporting member stations',
            'annual_percentiles': ANNUAL_PCTS, 'doy_window_days': 2 * DOY_WINDOW + 1}
    files['annual'] = out_dir / f'ca_normals_annual_{tag}.csv'
    annual.to_csv(files['annual'], index=False)
    files['doy'] = out_dir / f'ca_normals_doy_{tag}.csv'
    doy.to_csv(files['doy'], index=False)
    st = stations.copy()
    files['stations'] = out_dir / f'ca_normals_stations_{tag}.csv'
    st.to_csv(files['stations'], index=False)
    if fdra_tbl is not None:
        ft = fdra_tbl.copy()
        ft['stations'] = ft['stations'].apply(lambda s: ', '.join(map(str, s)))
        files['fdra'] = out_dir / f'ca_fdra_stations_{tag}.csv'
        ft.to_csv(files['fdra'], index=False)
    files['meta'] = out_dir / f'ca_normals_meta_{tag}.json'
    files['meta'].write_text(json.dumps(meta, indent=2))
    for k, f in files.items():
        log.info('wrote %-9s %s (%.1f MB)', k, f.name, f.stat().st_size / 1e6)
    return files


# ── 7. Site climatology (season-tracking charts, R6 style) ────────────────────
# Per FDRA, from the 2016–2025 daily FDRA values (FDOP stations, FDRA fuel model):
#   • ERC / BI / SFDI annual-percentile lookups (101 quantiles) — the GitHub site turns
#     live values into "annual percentile" with these
#   • per day-of-year max / avg of the annual percentiles (chart envelopes) and every
#     year's series (for analog years)
#   • live fuel moisture (herb, woody) avg / min by day of year
#   • temperature / RH by day of year when FEMS weather history is available
#   • fire-season end: first day after which ERC stays below the 70th percentile
#     (searched from July 1), its distribution across years → 25/50/75/90 % dates
# SFDI follows Jolly et al. 2019: p = ERC′ × BI′ (annual percentiles), SFDI = percentile
# of p in the same climatology; classes Low ≤60 < Moderate ≤80 < High ≤90 < Very High ≤97 < Severe.
SFDI_BREAKS = [60, 80, 90, 97]
SFDI_LABELS = ['Low', 'Moderate', 'High', 'Very High', 'Severe']
SEASON_END_PCT = 70
SEASON_SEARCH_FROM_DOY = 182   # July 1


def _pct_of(values: np.ndarray, sorted_clim: np.ndarray) -> np.ndarray:
    """Annual percentile (0–100) of each value within a sorted climatology."""
    v = np.asarray(values, dtype=float)
    out = np.full(v.shape, np.nan)
    ok = ~np.isnan(v)
    if len(sorted_clim):
        out[ok] = np.searchsorted(sorted_clim, v[ok], side='right') / len(sorted_clim) * 100
    return out


def _doy365(dates: pd.Series) -> np.ndarray:
    doy = dates.dt.dayofyear.to_numpy()
    leap = dates.dt.is_leap_year.to_numpy()
    return np.where(leap & (doy > 59), doy - 1, doy)


def season_end_doy(doy: np.ndarray, pct: np.ndarray, thresh=SEASON_END_PCT, start=SEASON_SEARCH_FROM_DOY):
    """First DOY ≥ start after which pct stays < thresh for the rest of the year (None if never)."""
    s = pd.Series(pct, index=doy).groupby(level=0).mean().reindex(range(1, 366))
    s = s.loc[start:]
    above = s[s >= thresh]
    if s.dropna().empty:
        return None
    if above.empty:
        return int(start)
    last = int(above.index.max())
    return last + 1 if last < 365 else None


def fdra_site_climo(daily_unit: pd.DataFrame, weather: pd.DataFrame | None = None) -> dict:
    """Climatology bundle for one FDRA (daily_unit: one unit's rows from unit_daily)."""
    d = daily_unit.sort_values('date').copy()
    d['doy'] = _doy365(d['date'])
    d['year'] = d['date'].dt.year
    q = np.linspace(0, 100, 101)
    out = {'years': sorted(d['year'].unique().tolist()), 'n_days': int(len(d))}
    clim = {}
    for k in ('erc', 'bi'):
        v = np.sort(d[k].dropna().to_numpy())
        clim[k] = v
        out[f'{k}_q'] = np.round(np.percentile(v, q), 2).tolist() if len(v) else []
        d[f'{k}_p'] = _pct_of(d[k], v)
    prod = d['erc_p'] * d['bi_p']
    clim['prod'] = np.sort(prod.dropna().to_numpy())
    out['sfdi_prod_q'] = np.round(np.percentile(clim['prod'], q), 1).tolist() if len(clim['prod']) else []
    d['sfdi'] = _pct_of(prod, clim['prod'])

    days = np.arange(1, 366)
    env = {}
    for k in ('erc_p', 'bi_p', 'sfdi'):
        g = d.groupby('doy')[k]
        env[k] = {'max': g.max().reindex(days).round(1).tolist(), 'avg': g.mean().reindex(days).round(1).tolist()}
    for k in ('erc', 'bi'):
        g = d.groupby('doy')[k]
        env[k] = {'max': g.max().reindex(days).round(1).tolist(), 'avg': g.mean().reindex(days).round(1).tolist()}
    for k in ('herb', 'woody', 'fm100', 'fm1000'):
        g = d.groupby('doy')[k]
        env[k] = {'min': g.min().reindex(days).round(1).tolist(), 'avg': g.mean().reindex(days).round(1).tolist()}
    out['envelope'] = env
    # every year's percentile series (analog years)
    out['years_series'] = {
        str(y): {k: g.groupby('doy')[k].mean().reindex(days).round(0).tolist()   # Feb 29 folded into Feb 28
                 for k in ('erc_p', 'bi_p', 'sfdi')}
        for y, g in d.groupby('year')}
    # season end
    ends = {str(y): season_end_doy(g['doy'].to_numpy(), g['erc_p'].to_numpy()) for y, g in d.groupby('year')}
    out['season_end_doy'] = ends
    ev = np.array([v for v in ends.values() if v is not None])
    out['season_end_quantiles'] = ({str(p): int(np.percentile(ev, p)) for p in (25, 50, 75, 90)} if len(ev) else {})
    # weather (optional)
    if weather is not None and len(weather):
        w = weather.copy()
        w['doy'] = _doy365(w['date'])
        for k in ('tmax', 'rhmin'):
            if k in w:
                g = w.groupby('doy')[k]
                env[k] = {'avg': g.mean().reindex(days).round(1).tolist(),
                          ('max' if k == 'tmax' else 'min'): (g.max() if k == 'tmax' else g.min()).reindex(days).round(1).tolist()}
    out['sfdi_breaks'] = SFDI_BREAKS
    out['sfdi_labels'] = SFDI_LABELS
    return out


def _clean(o):
    """NaN/inf → None, numpy scalars → Python, recursively (valid JSON for the site)."""
    if isinstance(o, dict):
        return {k: _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.integer):
        return int(o)
    return o


def write_site_climo(daily: pd.DataFrame, fdra_attrs: pd.DataFrame, out_dir: Path,
                     weather_by_unit: dict | None = None) -> Path:
    """site_data/climo/<fdra_id>.json for every FDRA + site_data/fdras.json index."""
    cdir = out_dir / 'climo'
    cdir.mkdir(parents=True, exist_ok=True)
    idx = []
    f = daily[daily['unit_type'] == 'FDRA']
    for uid, g in f.groupby('unit_id'):
        w = (weather_by_unit or {}).get(uid)
        c = fdra_site_climo(g, w)
        safe = re.sub(r'[^A-Za-z0-9_-]+', '_', uid)
        (cdir / f'{safe}.json').write_text(json.dumps(_clean(c), separators=(',', ':'), allow_nan=False))
        row = {'fdra_id': uid, 'file': f'climo/{safe}.json', 'years': c['years'], 'n_days': c['n_days'],
               'season_end_quantiles': c['season_end_quantiles']}
        idx.append(row)
    (out_dir / 'fdras.json').write_text(json.dumps(idx, indent=1))
    log.info('site climatology: %d FDRAs → %s', len(idx), out_dir)
    return out_dir


# ── 8. FEMS weather history (daily Tmax / RHmin) — public, batched ──────────────
def download_weather(stations: pd.DataFrame, start_year: int, end_year: int, raw_dir: Path,
                     workers: int = 4):
    """Batched, cached FEMS wx daily summaries (TemperatureMax, RelativeHumidityMin)."""
    return _download(FEMS_WX_URL, 'wx', stations['stationId'], None, start_year, end_year, raw_dir, workers)


def load_weather_daily(stations: pd.DataFrame, start_year: int, end_year: int, raw_dir: Path) -> pd.DataFrame:
    """Daily max temperature and min RH per station (observed rows only)."""
    want = set(stations['stationId'].astype(int))
    frames = []
    for f in sorted(raw_dir.glob('wx_*.csv')):
        if f.stat().st_size < 50:
            continue
        df = pd.read_csv(f)
        df.columns = df.columns.str.strip()
        if 'TemperatureMax(F)' not in df.columns:
            continue
        df = df[df['ObservationType'].astype(str).str.upper() == 'O']
        frames.append(pd.DataFrame({
            'stationId': pd.to_numeric(df['StationId'], errors='coerce'),
            'date': pd.to_datetime(df['Date'], errors='coerce').dt.normalize(),
            'tmax': pd.to_numeric(df['TemperatureMax(F)'], errors='coerce'),
            'rhmin': pd.to_numeric(df['RelativeHumidityMin(%)'], errors='coerce')}))
    if not frames:
        return pd.DataFrame(columns=['stationId', 'date', 'tmax', 'rhmin'])
    w = pd.concat(frames, ignore_index=True).dropna(subset=['stationId', 'date'])
    w['stationId'] = w['stationId'].astype(int)
    w = w[w['stationId'].isin(want) & w['date'].dt.year.between(start_year, end_year)]
    return w.groupby(['stationId', 'date'], as_index=False)[['tmax', 'rhmin']].mean()


def unit_weather(wx: pd.DataFrame, membership: dict) -> dict:
    """{fdra_id: daily tmax/rhmin averaged over its stations}."""
    out = {}
    for uid, members in membership.items():
        sids = [m[0] if isinstance(m, tuple) else m for m in members]
        sub = wx[wx['stationId'].isin(sids)]
        if len(sub):
            out[uid] = sub.groupby('date')[['tmax', 'rhmin']].mean().reset_index()
    return out


In [ ]:
%%writefile fdra_station_membership_2026.csv
fdra_id,season,stationId,fuel_model
ANF-Front Country,,45443,X
ANF-Front Country,,45405,X
ANF-Front Country,,45411,X
ANF-Front Country,,45437,X
ANF-Front Country,,45421,X
ANF-High Country,,45401,X
ANF-High Country,,45436,X
ANF-High Country,,45435,X
ANF-Saugus,,45438,X
ANF-Saugus,,45449,X
ANF-Saugus,,45435,X
ANF-Saugus,,45423,X
ANF-Saugus,,45426,X
ANF-Saugus,,45448,X
BDF-660,,45601,X
BDF-660,,45113,X
BDF-660,,45108,X
BDF-660,,45109,X
BDF-670,,45604,Y
BDF-670,,45605,Y
BDF-670,,45612,Y
BDF-673,,45616,X
BDF-673,,45603,X
BDF-692,,45105,Y
BDF-692,,45101,Y
BDF-692,,45133,Y
BDF-694,,45102,X
BDF-694,,45114,X
BDF-694,,45111,X
CNF-Inland,,45704,X
CNF-Inland,,45710,X
CNF-Inland,,45729,X
CNF-Inland,,45734,X
CNF-Inland,,45744,X
CNF-Inland,,45745,X
CNF-Mountain,,45619,X
CNF-Mountain,,45635,X
CNF-Mountain,,45636,X
CNF-Mountain,,45707,X
CNF-Mountain,,45708,X
CNF-Mountain,,45709,X
CNF-Mountain,,45711,X
CNF-Mountain,,45740,X
ENF-335,peak,42601,Y
ENF-335,peak,42603,Y
ENF-335,peak,42608,Y
ENF-335,peak,42611,Y
ENF-335,peak,42615,Y
ENF-335,non-peak,42602,Y
ENF-335,non-peak,42603,Y
ENF-335,non-peak,42608,Y
ENF-335,non-peak,42611,Y
ENF-335,non-peak,42615,Y
ENF-PENDING-HIGH-ELEV,,42603,Y
ENF-PENDING-HIGH-ELEV,,42604,Y
INF-460,,43711,Y
INF-460,,44709,Y
INF-460,,43710,Y
INF-461,,43708,Y
INF-461,,43710,Y
INF-461,,44803,Y
INF-461,,44804,Y
INF-443,,44722,Y
KNF-200,,40225,Y
KNF-200,,40231,Y
KNF-200,,40246,Y
KNF-202,,40203,Y
KNF-202,,40222,Y
KNF-202,,40231,Y
KNF-204,,40218,Y
KNF-204,,40225,Y
KNF-204,,40237,Y
KNF-208,,40237,Y
KNF-208,,40239,Y
KNF-208,,40245,Y
KNF-214,,40240,Y
KNF-214,,40243,Y
KNF-220,,40240,Y
KNF-220,,40243,Y
LNF-ALL,,40904,Y
LNF-ALL,,40703,Y
LNF-ALL,,40731,Y
LNF-ALL,,40723,Y
LNF-ALL,,40609,Y
LPF-507,,44302,X
LPF-512,,45220,X
LPF-514,,45218,X
LPF-514,,45221,X
LPF-514IS,,45216,X
LPF-514IS,,45217,X
LPF-560,,44301,X
LPF-560,,44317,X
LPF-585,,45201,X
LPF-585,,44901,X
LPF-586,,45203,X
LPF-588,,45303,X
LPF-588,,44914,X
LPF-650,,45307,X
LPF-650,,45314,X
LPF-650,,45308,X
LPF-651,,45302,X
MDF-258,,40221,Z
MDF-258,,40233,Z
MDF-258,,40302,Z
MDF-258,,40306,Z
MDF-258,,40312,Z
MDF-258,,40313,Z
MDF-258,,40314,Z
MNF-162,,41005,Y
MNF-162,,41018,Y
MNF-162,,41402,Y
MNF-162,,41406,Y
MNF-165,,411018,Y
MNF-165,,41101,Y
MNF-170,,40802,Z
MNF-170,,40816,Z
MNF-170,,41402,Z
MNF-170,,41503,Z
PNF-West,,41214,Y
PNF-West,,41304,Y
PNF-West,,41701,Y
PNF-Central,,40910,Y
PNF-Central,,40916,Y
PNF-Central,,40915,Y
PNF-East,,40917,Y
PNF-East,,40921,Y
PNF-East,,40709,Y
SHF-230,,40501,Y
SHF-230,,40503,Y
SHF-230,,40512,Y
SHF-230,,40516,Y
SHF-230,,40517,Y
SHF-230,,40518,Y
SHF-230,,40520,Y
SHF-238,,40812,Y
SHF-238,,40511,Y
SHF-240,,40217,Y
SHF-240,,40247,Y
SHF-240,,40618,Y
SHF-241,,40614,Y
SHF-241,,40635,Y
SHF-243,,40221,Y
SHF-243,,40244,Y
SNF-528,,44110,Y
SNF-528,,44112,Y
SNF-528,,44204,Y
SNF-528,,42207,Y
SNF-528,,44503,Y
SNF-528,,44505,Y
SNF-528,,44510,Y
SNF-532,,44109,Y
SNF-532,,44203,Y
SNF-532,,44503,Y
SNF-532,,44505,Y
SNF-533,,43612,Y
SNF-533,,44511,Y
SNF-533,,44520,Y
SRF-110,,40102,Y
SRF-110,,40105,Y
SRF-110,,40101,Y
SRF-112-115,,40404,Y
SRF-112-115,,40408,Y
SRF-112-115,,40519,Y
SRF-112-115,,40430,Y
SRF-130,,40507,Y
SRF-130,,40508,Y
SRF-201,,40231,Y
SRF-201,,40246,Y
SRF-RWP,,40102,Y
SRF-RWP,,40425,Y
STF-348,,44115,Y
STF-348,,43605,Y
STF-360,,43210,Y
STF-370,,43615,Y
SQF-438,,44701,X
SQF-438,,44719,X
SQF-438,,44712,X
SQF-438,,44717,X
SQF-438,,44508,X
SQF-445,,45002,X
SQF-445,,45016,X
SQF-435,,44713,X
SQF-435,,44732,X
SQF-440,,44707,Y
SQF-440,,44722,Y
SQF-440,,44726,Y
SQF-440,,45009,Y
TNF-350,,41304,Y
TNF-350,,41806,Y
TNF-350,,41908,Y
TNF-354,,41901,Y
TNF-380,,41302,Y
TNF-380,,41310,Y
TNF-380,,41311,Y
TMU-335,,42616,Y
TMU-335,,41909,Y
TMU-335,,261205,Y
TMU-335,,260117,Y
KNF-210,,40247,Y
SNF-529,,44503,Y
SNF-529,,44510,Y
SNF-529,,44505,Y
SNF-exempt,,44110,Y
ENF-330,,41908,Y
ENF-340,,42603,Y

In [ ]:
# 5 — Stations, PSAs and FDRA station lists
import logging, importlib
import pandas as pd
logging.basicConfig(level=logging.INFO, format='[%(asctime)s] %(message)s', datefmt='%H:%M:%S', force=True)
import ca_normals as cn; importlib.reload(cn)
cn.BATCH = BATCH

stations = cn.load_ca_stations()                  # live CA NFDRS 3.0 layer (public)
psas     = cn.load_ca_psas()
fdra_mem, fdra_pairs = cn.load_fdra_membership(FDRA_MEMBERSHIP)

downloads = pd.concat([stations[['stationId', 'FuelModel']], fdra_pairs], ignore_index=True).drop_duplicates()
n_ids = downloads['stationId'].nunique()
n_batches = -(-n_ids // BATCH) * (END_YEAR - START_YEAR + 1)
print(f'{len(stations)} RAWS, {stations.PSA.nunique()} PSAs, {len(fdra_mem)} FDRAs')
print(f'{n_ids} stations, fuel models {sorted(downloads.FuelModel.unique())} → {n_batches} FEMS requests')

In [ ]:
# 6 — Download FEMS history in batches of 20 (no login; ~5–10 min the first time)
failed = cn.download_history(downloads, START_YEAR, END_YEAR, RAW_DIR, workers=WORKERS)
print('Failed batch-years:', len(failed), '→ run this cell again to retry' if failed else '✓ all downloaded')

In [ ]:
# 7 — Load observed history and check coverage
hist = cn.load_history(downloads, START_YEAR, END_YEAR, RAW_DIR)
cov = (hist.assign(year=hist['date'].dt.year)
           .groupby('year').agg(stations=('stationId', 'nunique'), station_days=('erc', 'count')))
display(cov)
low = (hist.groupby(['stationId', 'fm'])['erc'].count() / (365.25 * (END_YEAR - START_YEAR + 1)))
print(f'Stations with < 50% of days: {(low < 0.5).sum()}  →', low[low < 0.5].sort_values().head(15).round(2).to_dict())

In [ ]:
# 8 — Aggregate and compute normals
membership = {
    'STATION': {str(s): [s] for s in stations['stationId']},          # fuel model Y
    'PSA':     stations.groupby('PSA')['stationId'].apply(list).to_dict(),
}
if fdra_mem:
    membership['FDRA'] = fdra_mem                                       # per-FDRA fuel model

daily = pd.concat([cn.unit_daily(hist, m, ut) for ut, m in membership.items()], ignore_index=True)
annual = cn.annual_normals(daily)

doy_src = daily if INCLUDE_STATION_DOY else daily[daily['unit_type'] != 'STATION']
doy = cn.doy_normals(doy_src, plot_year=END_YEAR + 1)
print(annual.groupby('unit_type').size().to_dict(), '| DOY rows:', len(doy))
annual[annual['unit_type'] == 'PSA'][['unit_id', 'n_days', 'erc_mean', 'erc_p90', 'erc_p97',
                                     'fm100_p3', 'fm100_p10', 'ic_p90', 'sc_p90']].head(12)

In [ ]:
# 9 — Write outputs
files = cn.write_outputs(OUT_DIR, START_YEAR, END_YEAR, stations, None, annual, doy)
for k, f in files.items():
    print(f'{k:9s} {f}')

In [ ]:
# 10 — Quick look: ERC and FM-100 normals for one PSA
import matplotlib.pyplot as plt
PSA = 'SC10'
d = doy[(doy.unit_type == 'PSA') & (doy.unit_id == PSA)]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, k in zip(axes, ['erc', 'fm100']):
    ax.fill_between(d.plot_date, d[f'{k}_p10'], d[f'{k}_p90'], alpha=.25, label='p10–p90')
    ax.plot(d.plot_date, d[f'{k}_p50'], lw=1.5, label='median')
    ax.set_title(f'{PSA} {k.upper()} normals {START_YEAR}–{END_YEAR}'); ax.legend()
plt.tight_layout(); plt.show()

## Season-tracking climatology for the FDRA website

These cells write `normals/output/site_data/`, the files the GitHub-hosted FDRA site uses for its
charts. For each FDRA:
- **ERC, BI and SFDI** annual-percentile lookups, so the site converts live values the same way;
- daily **max and average** envelopes, plus each year's series for analog years;
- **live fuel moisture** (herb, woody) average and minimum by day of year;
- **fire-season end** dates (ERC drops below the 70th percentile and stays there, searched from July 1),
  with the 25/50/75/90% dates;
- **temperature and RH** (daily max temperature, min RH) from the FEMS weather daily summary (cell 11).

SFDI follows Jolly et al. (2019): ERC′ × BI′, re-ranked against the same climatology; classes break at
60 / 80 / 90 / 97. **Precaution Value (PV)** is left out until its definition is confirmed.

In [ ]:
# 11 — Weather daily summary (max temperature / min RH) for the TRH chart — batches of 20, no login
wx_stations = fdra_pairs[['stationId']].drop_duplicates()
failed_wx = cn.download_weather(wx_stations, START_YEAR, END_YEAR, RAW_DIR, workers=WORKERS)
wx = cn.load_weather_daily(wx_stations, START_YEAR, END_YEAR, RAW_DIR)
print('weather station-days:', len(wx), '| failed batch-years:', len(failed_wx))
wx_by_unit = cn.unit_weather(wx, fdra_mem) if len(wx) else None

In [ ]:
# 12 — Write the site climatology (site_data/)
SITE_DIR = OUT_DIR / 'site_data'
cn.write_site_climo(daily, None, SITE_DIR, wx_by_unit)
import json
idx = json.loads((SITE_DIR / 'fdras.json').read_text())
print(len(idx), 'FDRAs written to', SITE_DIR)
pd.DataFrame(idx)[['fdra_id', 'n_days', 'season_end_quantiles']].head(15)

In [ ]:
# 13 — Preview one FDRA the way the site will draw it
import matplotlib.pyplot as plt, numpy as np, json
FDRA = 'ENF-335 (peak)' if (SITE_DIR / 'climo' / 'ENF-335_peak_.json').exists() else idx[0]['fdra_id']
c = json.loads((SITE_DIR / [r['file'] for r in idx if r['fdra_id'] == FDRA][0]).read_text())
days = np.arange(1, 366)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, k, t in zip(axes, ['erc_p', 'bi_p', 'sfdi'], ['ERC annual percentile', 'BI annual percentile', 'SFDI']):
    env = c['envelope'][k]
    ax.plot(days, env['max'], color='#d62728', lw=1.5, label='max')
    ax.plot(days, env['avg'], color='#555', lw=2, label='avg')
    last = str(max(map(int, c['years_series'])))
    ax.plot(days, c['years_series'][last][k], color='k', lw=1, label=last)
    for y in (60, 80, 90, 97):
        ax.axhline(y, ls=':', color='#999', lw=.8)
    ax.set_title(f'{FDRA} — {t}'); ax.set_ylim(0, 100); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print('Season end (25/50/75/90 %):', c['season_end_quantiles'])

## What to do with the outputs

1. **Site data:** commit `normals/output/site_data/` to the repo (it's public; values only, no thresholds).
   The daily GitHub Action reads it to draw the season-tracking charts.
2. **Optional AGOL tables:** `ca_normals_annual_2016_2025.csv` and `ca_normals_doy_2016_2025.csv` can be
   added as hosted tables (New item → Your device → hosted table) if you want normals in Experience Builder too.